In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from functools import reduce

In [0]:
customers_df = spark.read.format("csv").option("inferSchema",True).option("header",True).load("/Workspace/Users/tekiyaswanthkumar@gmail.com/databricks_practice/pyspark_customers.csv")

In [0]:
transactions_df = spark.read.format("CSV").option("inferSchema",True).option("header",True).load("/Workspace/Users/tekiyaswanthkumar@gmail.com/databricks_practice/pyspark_transactions.csv")

In [0]:
transactions_df.printSchema()

In [0]:
customers_df.printSchema()

In [0]:
customers_df.count()

In [0]:
region_customers = customers_df.groupBy("region").agg(count('customer_id'))
active_customers = customers_df.groupBy("is_active").agg(count('customer_id'))
loyality_points_per_segment = customers_df.groupBy("segment").agg(avg('loyalty_points').alias("avg_loyalty_points"))
top_customers_cities = customers_df.groupBy("city").agg(count("customer_id").alias("count_of_customers")).orderBy(desc(col("count_of_customers"))).limit(5)


In [0]:
customers_df.filter(reduce(lambda x,y : x.isNull() | y.isNull(), [col(x) for x in customers_df.columns])).display()

In [0]:
transactions_df.limit(5).display()

In [0]:
# transactions_df.groupBy("status").agg(count('*').alias("count_of_transactions")).display()
# transactions_df.withColumn("net_price",round(col("gross_amount")*(1-col("discount_pct")), 3)).filter("status ='COMPLETED'").groupBy("product_category").agg(round(sum("net_price"),3).alias("net_revenue")).display()
transactions_df.filter("status = 'COMPLETED'").groupBy("payment_method").agg(round(avg("gross_amount"),2).alias("avg_gross")).orderBy(col("avg_gross").desc()).display()

In [0]:
customers_df.limit(2).display()
transactions_df.limit(2).display()

In [0]:
# cleansed_transactions_df = transactions_df.filter("status = 'COMPLETED'").dropDuplicates().dropna(subset=["customer_id"]).withColumn("net_price",col("gross_amount")*(1-col("discount_pct"))).groupBy("customer_id").agg(count('*').alias("total_transactions"), sum(col("quantity")).alias("total_quantity"), sum(col("gross_amount")).alias("total_gross_revenue"), sum("net_price").alias("total_net_revenue"), avg("net_price").alias("avg_transaction_value"))

from pyspark.sql.functions import *

cleansed_transactions_df = (
    transactions_df
    .filter("status = 'COMPLETED'")
    .dropDuplicates()
    .dropna(subset=["customer_id"])
    .withColumn("net_price", col("gross_amount") * (1 - col("discount_pct")))
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_transactions"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 3).alias("total_gross_revenue"),
        round(sum("net_price"), 3).alias("total_net_revenue"),
        round(avg("net_price"), 3).alias("avg_transaction_value")
    )
)


In [0]:
customers_df.join(cleansed_transactions_df, on = "customer_id", how = 'inner').select("customer_id", "customer_name","region", "total_transactions", "total_quantity", "total_gross_revenue", "total_net_revenue", "avg_transaction_value").orderBy(desc("total_net_revenue")).limit(10).display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

cdc_data = [
    (100001, "UPDATE", "Ravi Kumar", "Hyderabad", "ravi@gmail.com", "South", "Premium", 1200, "2026-11-13 09:00:00"),
    (100002, "UPDATE", "Suresh Reddy", "Chennai", "suresh.new@gmail.com", "South", "Standard", 800, "2026-11-13 09:05:00"),
    (100003, "UPDATE", "Priya Sharma", "Bangalore", "priya@gmail.com", "South", "Premium", 1500, "2026-11-13 09:10:00"),
    (100004, "INSERT", "Kiran Kumar", "Visakhapatnam", "kiran@gmail.com", "South", "Standard", 500, "2026-11-13 09:15:00"),
    (100005, "UPDATE", "Arjun Rao", "Pune", "arjun@gmail.com", "West", "Premium", 2500, "2026-11-13 09:20:00")
]

columns = [
    "customer_id", "operation", "name", "city", "email",
    "region", "segment", "loyalty_points", "updated_at"
]

cdc_source_df = spark.createDataFrame(cdc_data, columns)

cdc_source_df = cdc_source_df.withColumn(
    "updated_at",
    to_timestamp("updated_at")
)

display(cdc_source_df)

In [0]:
from delta.tables import DeltaTable

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("chatgpt_questions.problems.dim_customers")

In [0]:
%sql
select count(*) from chatgpt_questions.problems.dim_customers

In [0]:
"""
src = cdc_data
tgt = chatgpt_questions.problems.dim_customers


"""

from delta.tables import DeltaTable

cdc_source_df = cdc_source_df.withColumn("rn",row_number().over(Window.partitionBy("customer_id").orderBy(desc(col("updated_at"))))).filter("rn = 1")
display(cdc_source_df)

target = DeltaTable.forName(spark,"chatgpt_questions.problems.dim_customers")
target.alias("tgt").merge(cdc_source_df.alias("src"), "tgt.customer_id = src.customer_id").whenMatchedUpdate(condition = "src.operation = 'UPDATE'",set={  
    "customer_name":"src.name", 
    "city": "src.city", 
    "email": "src.email", 
    "region": "src.region", 
    "segment": "src.segment", 
    "loyalty_points": "src.loyalty_points"
}).whenNotMatchedInsert(condition = "src.operation = 'INSERT'",values = {
    "customer_name":"src.name", 
    "city": "src.city", 
    "email": "src.email", 
    "region": "src.region", 
    "segment": "src.segment", 
    "loyalty_points": "src.loyalty_points"
}).execute()

In [0]:
%sql
select * from chatgpt_questions.problems.dim_customers_scd2 limit 1

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

cdc_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("region", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("loyalty_points", IntegerType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("operation", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])

from datetime import datetime, date

cdc_data = [
    (100001, "Customer_100001", "user100001@example.com", "Hyderabad", "South", "Corporate", date(2021, 8, 24), 15976, True, "UPDATE", datetime(2026, 11, 14, 10, 0, 0)),
    (100002, "Customer_100002", "user100002@example.com", "Chennai", "South", "Consumer", date(2024, 11, 26), 9587, True, "UPDATE", datetime(2026, 11, 14, 10, 5, 0)),
    (100004, "Customer_100004", "user100004@example.com", "Mumbai", "West", "Enterprise", date(2024, 4, 12), 8228, True, "INSERT", datetime(2026, 11, 14, 10, 10, 0))
]

cdc_source_df = spark.createDataFrame(cdc_data, schema=cdc_schema)

# display(cdc_source_df)

cdc_source_df = spark.createDataFrame(cdc_data, schema=cdc_schema)

cdc_source_df = cdc_source_df.withColumn(
    "signup_date", to_date("signup_date")
).withColumn(
    "updated_at", to_timestamp("updated_at")
)

print(cdc_source_df.columns)

In [0]:
# deduplication of source
cdc_source_df = cdc_source_df.filter("is_active = 'true'").withColumn("rn",row_number().over(Window.partitionBy("customer_id").orderBy(desc(col("updated_at"))))).filter("rn = 1").drop("rn")
tgt_df = spark.read.table('chatgpt_questions.problems.dim_customers_scd2').filter("is_current = 'true'")

cdc_source_df.alias("src_df").join(tgt_df.alias("tgt_df"), on = col('src_df.customer_id') == col('tgt_df.customer_id'), how = "left").display()

















"""

from delta.tables import DeltaTable

tgt_df.display()

#existing customers whose attributes have changed
cdc_source_df = cdc_source_df.alias("cdc_source_df").join(tgt_df.alias("tgt_df"),on = cdc_source_df.customer_id == tgt_df.customer_id,how ='left').where("not(cdc_source_df.city <=> tgt_df.city) or not(cdc_source_df.region <=> tgt_df.region) or not(cdc_source_df.segment <=> tgt_df.segment)").select("cdc_source_df.*")
#3 customers attributes changed


tgt_tbl = DeltaTable.forName(spark,"chatgpt_questions.problems.dim_customers_scd2")
tgt_tbl.alias("tgt").merge(cdc_source_df.alias("src"), "tgt.customer_id = src.customer_id").whenMatchedUpdate("src.operation = 'UPDATE'", set = {
"effective_end_date": current_timestamp(),
"is_current": False
}).whenNotMatchedInsert("src.operation = 'INSERT'", values = {
"customer_id": "src.customer_id",
"customer_name": "src.name",
"email": "src.email",
"city": "src.city",
"region": "src.region",
"segment": "src.segment",
"signup_date": "src.signup_date",
"loyalty_points": "src.loyalty_points",
"is_active": True,
"effective_start_date": current_timestamp(),
"effective_end_date": lit(None),
"is_current": True
}).execute()


#newCustomers
# cdc_source_df.alias("cdc_source_df").join(tgt_df.alias("tgt_df"), on = cdc_source_df.customer_id == tgt_df.customer_id,how ='leftanti').select("cdc_source_df.*").display()
#no new customers

# Existing customers whose values haven't changed
# cdc_source_df.alias("cdc_source_df").join(tgt_df.alias("tgt_df"),on = cdc_source_df.customer_id == tgt_df.customer_id,how ='left').where("(cdc_source_df.city <=> tgt_df.city) and (cdc_source_df.region <=> tgt_df.region) and (cdc_source_df.segment <=> tgt_df.segment)").select("cdc_source_df.*").display()
#0 """